# Quality and Utility Evaluation

In [1]:
import os
from pathlib import Path


# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    """Change the working directory to the repository root."""
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


set_project_root()

import json
from logging import INFO

import pandas as pd
from midst_toolkit.common.logger import log
from midst_toolkit.evaluation.quality import (
    AlphaPrecision,
    CorrelationMatrixDifference,
    KolmogorovSmirnovAndTotalVariation,
    MeanConfidenceIntervalOverlap,
    MeanF1ScoreDifference,
    MeanRegressionDifference,
)
from midst_toolkit.evaluation.quality.confidence_interval_overlap import ConfidenceLevel

from implementations.tabular_data.evaluation.display_utils import log_metrics

# Local imports
from implementations.tabular_data.evaluation.preprocessing import (
    get_numerical_and_categorical_column_names,
    preprocess_data_for_alpha_precision_eval,
    syntheval_preprocess,
)

## Load the data for quality, utility and privacy evaluation
Evaluation also needs a JSON file containing meta information about the dataset. The provided `meta_info.json` JSON file should provide information about the columns as well as the target column, specifically which columns correspond to numerical and categorical values, which column corresponds to a label (if any), and the downstream task to be performed with this dataset.
The provided pre-processing script (`preprocess_berka_trans.py`), generates this JSON file for the transaction table in the Berka dataset, and the evaluation pipeline loads it as a dictionary. Here is an example of `meta_info`.
```bash
}
    "num_col_idx": [0,3,4,7],
    "cat_col_idx": [1,2,5,6],
    "target_col_idx": [1],
    "task_type": "multiclass"
}
```
Types of supported tasks: "binclass","multiclass", and "regression"

Other dataframes to load:

- `real_train_data`: real data that is used to train the generative model

- `real_holdout_data`: real data that is NOT used in training

- `synthetic_data`: the synthesized data 



In [2]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data" / "single_table"
# Set data and output directories
base_data_dir = IMPLEMENTATION_ROOT / "data"
base_output_dir = IMPLEMENTATION_ROOT / "results"

TABLE_NAME = "trans"

real_train_data = pd.read_csv(base_data_dir / f"{TABLE_NAME}.csv")
real_holdout_data = pd.read_csv(base_data_dir / f"{TABLE_NAME}_holdout.csv")
synthetic_data = pd.read_csv(base_output_dir / "single_table_synthesizing/trans/_final" / f"{TABLE_NAME}_synthetic.csv")

with open(base_data_dir / "meta_info.json", "r") as f:
    meta_info = json.load(f)

log(INFO, f"Loaded {TABLE_NAME} data for evaluations")
log(INFO, f"Loaded meta_info for {TABLE_NAME} data")
log(INFO, f"Loaded {len(real_train_data)} rows of real training data")
log(INFO, f"Loaded {len(real_holdout_data)} rows of real holdout data")
log(INFO, f"Loaded {len(synthetic_data)} rows of synthetic data")

INFO :      Loaded trans data for evaluations
INFO :      Loaded meta_info for trans data
INFO :      Loaded 16000 rows of real training data
INFO :      Loaded 4000 rows of real holdout data
INFO :      Loaded 16000 rows of synthetic data


## Pre-process the data for evaluation

Preprocessing can have a noticeable impact on the metrics. As such, it is important to handle such transformations with care and to take into account the calculations being performed in the metrics themselves. For example, if computing distance-based metrics with Euclidean norms, categorical variables should be one-hot encoded, rather than ordinally encoded. This notebook handles proper preprocessing of the datasets before applying the various metrics in the pipeline.


In this notebook:

- When applying the `AlphaPrecision` metric, categorical variables are one-hot encoded and numerical variables are left untouched.

- For all other metrics, which leverage the `SynthEval` library to perform at least some portion of the computations, the standard SynthEval preprocessing pipeline is applied. This process ordinally encodes categorical variables and min-max encodes numerical variables. `SynthEval` library: https://github.com/schneiderkamplab/syntheval

In [3]:
# Shared preprocessing for syntheval based metrics if they are to be run
log(INFO, "Preprocessing Data with SynthEval pipeline")
numerical_columns, categorical_columns = get_numerical_and_categorical_column_names(real_train_data, meta_info)
# Categorical values are ordinal encoded, numerical values are min-max scaled
syntheval_real_data_train, syntheval_synthetic_data, syntheval_real_data_holdout = syntheval_preprocess(
    numerical_columns,
    categorical_columns,
    real_train_data,
    synthetic_data,
    real_holdout_data,
)

INFO :      Preprocessing Data with SynthEval pipeline


## Fidelity Evaluations

### Fidelity metric: $\alpha$-Precision and $\beta$-Recall
Compute several quality metrics based on the Alpha Precision measure originally proposed in
https://arxiv.org/abs/2301.07573 comparing the quality of synthetically generated data to real data.
Note that these metrics can be evaluated for each synthetic data point (which are useful for auditing and post-processing). Here we average the scores to reflect the overall quality of the data.




$\alpha$-Preicison and $\beta$-Recall are generalizations of Precision and Recall metrics proposed by [Sajjadi et al.](https://proceedings.neurips.cc/paper/2018/hash/f7696a9b362ac5a51c3dc8f098b73923-Abstract.html) in 2018. These metrics can range between $[0, 1]$ and the closest they are to $1$ the better.


- `Preicison`: measures the **fidelity or quality** of sythetic data. In more clear terms, it computes the proporation of synthetic datapoints that are *close* to real datapoints.  
- `Recall`: measures the **diversity** of synthetic data; i.e. the extent to which these samples cover the full variability of real samples. More clearly, recall computes the proportion of real datapoints that are *close* to synthetic datapoints.  


If we denote the distibution of real datapoints by $P(X)$ and the distribution of sythetic datapoints by $Q(Y)$, precision is the portion of $Q(Y)$ that can be generated by $P(X)$, while recall is the portion of $P(X)$ that can be generated by $Q(Y)$.


To better understand these concepts, let's assume that the real/generated dataponits are samples from an underlying real/generated manifold.
Precision measures the proportation of generated datapoints that fall on the real manifold, while recall measures the proportion of real datapoints that fall on the generated manifold.



**Outliers**: The Precision-Recall metrics are very sensitive to outliers since even a few outliers can greatly change the shape of the underlying manifold.
To address this limitation, $\pmb{\alpha}$**-Precision** and $\pmb{\beta}$**-Recall** are defined by assuming that a fraction $1−\alpha$ (or $1−\beta$) of the real (and synthetic) data are “outliers”, and $\alpha$ (or $\beta$) are “typical”. 
$\alpha$-Precision is the fraction of synthetic samples that resemble the “most typical” fraction $\alpha$ of real samples, whereas $\beta$-Recall is the fraction of real samples covered by the most typical fraction $\beta$ of synthetic samples.
The two metrics are evaluated for all $\alpha, \beta \in [0, 1]$, providing entire precision and recall curves instead of single numbers.


To illustrate, consider the below image. Blue and red points are real and generated datapoints, respectively. The large blue and red spheres show the underlying manifold that was estimated from real and generated datapoints.
Good quality generated datapoints should fall within the blue sphere like image *(c)*. They should not lie far from the blue sphere like *(a)*. Moreover, they should not be placed too close (or *copied*) to a real datapoint like *(b)*.
Image *(d)* shows an outlier in the real datapoints which is cut outside of the manifold due to the application on $\alpha$ and $\beta$.
If we used vanilla Precision and Recall, the blue sphere's radius should have increased to include the outlier which would also lead it to include noisy synthetic datapoints like *(a)*.

<div align="center">
  <img src="../images/alpha-precision.png" alt="alpha-precision image" width="530" height="400">
</div>



In [4]:
# The implementation is based heavily on the Synthcity library (https://github.com/vanderschaarlab/synthcity).
# Specifically, this class computes the alpha-precision, beta-recall,
# and authenticity scores between the two datasets.

# Step 1
log(INFO, "Preprocessing Data for Alpha Precision Evaluation")
# Categorical values are one-hot encoded, numerical values are left alone.
alpha_precision_real_data, alpha_precision_synthetic_data = preprocess_data_for_alpha_precision_eval(
    real_data=real_train_data,
    synthetic_data=synthetic_data,
    meta_info=meta_info,
)
# NOTE: Synthcity requires that the real and synthetic dataframes have the
# SAME number of datapoints.
if len(alpha_precision_real_data) > len(alpha_precision_synthetic_data):
    alpha_precision_real_data = alpha_precision_real_data.iloc[: len(alpha_precision_synthetic_data)]
    log(INFO, f"Truncated real data to {len(alpha_precision_synthetic_data)} rows")
else:
    alpha_precision_synthetic_data = alpha_precision_synthetic_data.iloc[: len(alpha_precision_real_data)]
    log(INFO, f"Truncated synthetic data to {len(alpha_precision_real_data)} rows")

# Step 2
log(INFO, "Running Alpha-Precision Evaluation")
metric = AlphaPrecision(naive_only=False)
results = metric.compute(alpha_precision_real_data, alpha_precision_synthetic_data)
log_metrics("Alpha Precision", results)

INFO :      Preprocessing Data for Alpha Precision Evaluation
INFO :      Truncated synthetic data to 16000 rows
INFO :      Running Alpha-Precision Evaluation
/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/midst_toolkit/evaluation/quality/synthcity/one_class.py:158: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.X = torch.tensor(x_train.reshape((-1, self.input_dim))).float()
INFO :      delta_precision_alpha_OC: 0.9252795977011494
INFO :      delta_coverage_beta_OC: 0.5713375
INFO :      authenticity_OC: 0.592625
INFO :      delta_precision_alpha_naive: 0.9865814655172414
INFO :      delta_coverage_beta_naive: 0.43795000000000006
INFO :      authenticity_naive: 0.4673125
INFO :      
Alpha Precision
--------------------------------------------------------------------------------

INFO :      Metric: delta_pr

## Fidelity: Density Estimation of columns

### Single-column similarity score
Essentially, each column in a table represents a single feature which accepts various values from a certain type; e.g. numerical, categorical, datetime or boolean. This feature can be described as a random variable where the values listed under the column are its samples. As a result, the density distribution of each column can be computed and compared between the real data and the synthetic data.


The better the distributions match each other, the better the quality of synthetic data.

The similarity of the two distriutions can be measured via different metrics for different data types; for example, **KSComplement (Kolmogorov-Smirnov Complement)** is used for numerical features and **TVComplement (Total Variation Distance Complement)** for categorical features. 


Both are performed as two-sided hypothesis tests to determine whether it
is likely that the distribution of a given column is the same between the two dataframes (null).
The main score is the average test statistic across all evaluated columns. Smaller is better. 


### Details:

#### **KST (Kolmogorov-Smirnov Test)**  
Computes the cumulative distribution function (CDF) of a numerical random variable for real and synthetic data. Then, finds the maximum different between the two CDFs $M$. Finally, the *KSComplement* score is defined as $1 - M$ so that the higher the score, the more similar the distributions.


The graphs below show two examples with real and synthetic data (black and green). At the left, the synthetic data is similar to the real data so the score is close to 1. At the right, the shapes are different so the score is lower.

<div align="center">
  <img src="../images/KSComplement.png" alt="KST (Kolmogorov-Smirnov Test)" width="600" height="200">
</div>


Source: https://docs.sdv.dev/sdmetrics/data-metrics/quality/kscomplement




**TVD (Total Variation Distance)**  
Computes the frequency of each category's appearance under a certain column and defines it as the said categorie's probability. Then, it computes the sum of differences of probabilities between real and synthetic data as 
$\delta(R, S) = \frac{1}{2} \sum_{\omega \in \Omega}  | R_\omega - S_\omega |$
.

Here, $\omega$ describes all the possible categories in a column, $\Omega$. Meanwhile, $R$ and $S$ refer to the real and synthetic frequencies for those categories. The *TVComplement* returns $1-TVD$ so that a higher score means higher quality.

The bar graph below compares real and synthetic data. Because of the differences between the categories, the TVComplement score is less than 1.0.

<div align="center">
  <img src="../images/TVD.png" alt="TVD (Total Variation Distance)" width="600" height="200">
</div>


### Metric results:
`KolmogorovSmirnovAndTotalVariation.compute()` compares each column in real vs synthetic data with a two-sided test of “same distribution?” Numerical columns use a Kolmogorov–Smirnov (KS) test; categorical columns use Total Variation Distance (TVD) plus a permutation test for the p-value.

**Important:** these are *raw distances*, not the KSComplement / TVComplement scores above. Here **smaller is better** (0 = identical distributions). Complement scores are `1 - distance`.

To compute each return key: the metric first computes the distance of **every column**, then averages those distances.

- **`avg stat` / `stat err`**: Mean (and standard error) of all column-level distances — KS for numbers and TVD for categories mixed together. This is the headline “how far apart are the univariate distributions?” number.
- **`avg ks` / `ks err`**: Same idea, but **only numerical columns**. KS is the largest vertical gap between the two CDFs (the $M$ in the figure above).
- **`avg tvd` / `tvd err`**: Same idea, but **only categorical columns**. TVD is $\frac{1}{2}\sum |R_\omega - S_\omega|$: how much the category frequencies disagree.
- **`avg pval` / `pval err`**: Mean (and SE) of the per-column p-values. A **small p-value** means “this gap is unlikely if the two columns really came from the same distribution.” Averaging p-values is a rough summary; the next keys are easier to interpret.
- **`num sigs` / `frac sigs`**: Count and fraction of columns whose p-value is below `significance_level` (default 0.05) — i.e. columns we treat as **significantly different**. `sigs cols` lists those column names.

Standard error (`* err`) is not “measurement noise”; it is how much the per-column scores vary around their mean. A large `stat err` means some columns match well and others do not.

In [6]:
log(INFO, "Running Kolmogorov-Smirnov and Total Variation Evaluation")
metric = KolmogorovSmirnovAndTotalVariation(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    significance_level=0.05,
    permutations=10,
    # Already preprocessing above
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data)
log_metrics("Kolmogorov-Smirnov and Total Variation", results)

INFO :      Running Kolmogorov-Smirnov and Total Variation Evaluation
INFO :      
Kolmogorov-Smirnov and Total Variation
--------------------------------------------------------------------------------

INFO :      Metric: avg stat	Score: 0.025668750000000004
INFO :      Metric: stat err	Score: 0.016157121418886557
INFO :      Metric: avg ks	Score: 0.04068750000000001
INFO :      Metric: ks err	Score: 0.03251926512341056
INFO :      Metric: avg tvd	Score: 0.01065
INFO :      Metric: tvd err	Score: 0.0032198602454143873
INFO :      Metric: avg pval	Score: 0.47286017975542893
INFO :      Metric: pval err	Score: 0.1429605143929449
INFO :      Metric: num sigs	Score: 1
INFO :      Metric: frac sigs	Score: 0.125
INFO :      Metric: sigs cols	Score: ['amount']
INFO :      --------------------------------------------------------------------------------



In [ ]:
# Added

import pandas as pd

quantiles = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

comparison = pd.DataFrame({
    "Real Train": real_train_data["amount"].quantile(quantiles),
    "Synthetic": synthetic_data["amount"].quantile(quantiles),
})

comparison["Relative Error"] = (
    (comparison["Synthetic"] - comparison["Real Train"]).abs()
    / comparison["Real Train"].abs().replace(0, float("nan"))
)

display(comparison)

,Real Train,Synthetic,Relative Error
0.25,130.075,125.413176,0.035840
0.50,2094.500,2001.278949,0.044508
0.75,6900.000,6756.955222,0.020731
0.90,18500.600,18628.391232,0.006907
0.95,25762.950,25659.895401,0.004000
0.99,45661.250,45793.773285,0.002902


In [ ]:
# Added

comparison_holdout = pd.DataFrame({
    "Real Holdout": real_holdout_data["amount"].quantile(quantiles),
    "Synthetic": synthetic_data["amount"].quantile(quantiles),
})

comparison_holdout["Relative Error"] = (
    (comparison_holdout["Synthetic"] - comparison_holdout["Real Holdout"]).abs()
    / comparison_holdout["Real Holdout"].abs().replace(0, float("nan"))
)

display(comparison_holdout)

,Real Holdout,Synthetic,Relative Error
0.25,132.80,125.413176,0.055624
0.50,2154.50,2001.278949,0.071117
0.75,6900.00,6756.955222,0.020731
0.90,17901.90,18628.391232,0.040582
0.95,26019.75,25659.895401,0.013830
0.99,46901.00,45793.773285,0.023608


## Fidelity: Confidence Interval Overlap (CIO)

While KS/TVD compare whole column distributions, this metric asks a narrower question: for each **numerical** column, is the **mean** of the synthetic data compatible with the mean of the real data? Compatibility is measured by how much the confidence intervals (CIs) around the two means overlap. Categorical columns are ignored.

**More overlap is better**: 1 means the intervals coincide, 0 means they do not touch at all.

Each CI is built around the column mean using the standard error of the mean and a **z**-score (not a t-score):

$$\text{CI} = \bar{x} \pm z \cdot \frac{s}{\sqrt{n}}, \qquad z = 1.96 \ \text{at}\ 95\%$$

where $\bar{x}$ is the column mean, $s$ its standard deviation, and $n$ the number of rows. Since the interval shrinks as $n$ grows, on large datasets even a small shift in the mean can drive the overlap to 0.

### Per-column overlap score

For the real CI $[L_r, U_r]$ and synthetic CI $[L_s, U_s]$, the shared width is $w = \min(U_r, U_s) - \max(L_r, L_s)$. If $w \le 0$ the intervals miss and the score is 0. Otherwise

$$J = \frac{1}{2}\left(\frac{w}{U_r - L_r} + \frac{w}{U_s - L_s}\right)$$

which averages the fraction of *each* interval that is shared, so a wide interval is not rewarded just for being wide. Example: real $[1, 3]$ and synthetic $[2, 5]$ share $w = 1$, giving $J = \frac{1}{2}(1/2 + 1/3) \approx 0.42$.

### Metric results

- **`avg overlap`**: mean of $J$ over all numerical columns, in $[0, 1]$. Closer to 1 is better.
- **`overlap err`**: standard error of those per-column scores; a large value means some columns match and others do not.
- **`num non-overlaps`**: number of columns with $J = 0$.
- **`frac non-overlaps`**: fraction of columns with no overlap. Closer to 0 is better.

`avg overlap` of 0 with `frac non-overlaps` of 1 means every numerical column's synthetic mean falls outside the real mean's CI.

In [10]:
log(INFO, "Running Confidence Interval Overlap Evaluation")
metric = MeanConfidenceIntervalOverlap(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    confidence_level=ConfidenceLevel(95),
    # Already preprocessing above
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data)
log_metrics("Mean Confidence Interval Overlap", results)

INFO :      Running Confidence Interval Overlap Evaluation
INFO :      
Mean Confidence Interval Overlap
--------------------------------------------------------------------------------

INFO :      Metric: avg overlap	Score: 0.6750101655924704
INFO :      Metric: overlap err	Score: 0.10031045413453878
INFO :      Metric: num non-overlaps	Score: 0.0
INFO :      Metric: frac non-overlaps	Score: 0.0
INFO :      --------------------------------------------------------------------------------



## Fidelity: Mean Correlation Matrix Difference (Pair-wise Correlation)

KS/TVD and CIO check **one column at a time**. This metric checks **pairs of columns**: do the same relationships show up in synthetic data (e.g. does `amount` still rise with `balance`)?

Build a correlation matrix for real data and one for synthetic data. Each entry is “how strongly column $i$ is associated with column $j$.” Subtract the two matrices and take the **Frobenius norm** of the difference — the square root of the sum of squared cell-wise errors. **Closer to 0 is better** (identical pairwise structure). The score grows with the number of columns, so compare it only across runs with the same `corr_mat_dims`.

### How each pair is scored

With `compute_mixed_correlations=True` (as below), every pair of columns is included. Categories can stay as labels; they do not need one-hot encoding.

- **Number vs number** — Pearson $r \in [-1, 1]$: linear co-movement.
- **Category vs category** — Cramér’s $V \in [0, 1]$: association from a contingency table (0 = independent, 1 = one category determines the other).
- **Category vs number** — correlation ratio $\eta \in [0, 1]$: how much of the numeric column’s variance is explained by the category (e.g. does `type` predict `amount`?).

If `compute_mixed_correlations=False`, only the numerical–numerical block is used.

### Metric results

- **`corr_mat_diff`**: Frobenius distance between the two matrices. 0 is a perfect match.
- **`corr_mat_dims`**: matrix size (number of columns included). With mixed correlations this is all columns (here 8).


In [11]:
log(INFO, "Running Mean Correlation Matrix Difference Evaluation")
metric = CorrelationMatrixDifference(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    compute_mixed_correlations=True,  # Whether or not to compute correlations between the categorical variables and
    # the categorical and numerical variables.
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data)
log_metrics("Correlation Matrix Difference", results)

INFO :      Running Mean Correlation Matrix Difference Evaluation
INFO :      
Correlation Matrix Difference
--------------------------------------------------------------------------------

INFO :      Metric: corr_mat_diff	Score: 0.06283561471156647
INFO :      Metric: corr_mat_dims	Score: 8
INFO :      --------------------------------------------------------------------------------



In [12]:
# Added


columns = ["amount", "balance"]

real_corr = real_train_data[columns].corr()
synthetic_corr = synthetic_data[columns].corr()
holdout_corr = real_holdout_data[columns].corr()

comparison = pd.DataFrame({
    "Real Train": [real_corr.loc["amount", "balance"]],
    "Real Holdout": [holdout_corr.loc["amount", "balance"]],
    "Synthetic": [synthetic_corr.loc["amount", "balance"]]
}, index=["Amount vs Balance"])

comparison["Absolute Difference vs Train"] = (
    comparison["Synthetic"] - comparison["Real Train"]
).abs()

display(comparison)

,Real Train,Real Holdout,Synthetic,Absolute Difference vs Train
Amount vs Balance,0.454228,0.451836,0.451515,0.002713


## Utility: F1 Score Difference

Fidelity asks whether synthetic data *looks* like real data. **Utility** asks whether it is *useful*: can you train a classifier on synthetic rows and still predict a real label well?

Here the label is `trans_type` (multiclass). Four models are trained twice — random forest, AdaBoost, SVM, and logistic regression — once on **real** data and once on **synthetic** data. Both versions are then scored with **macro F1** on **real** test rows.

- **Train on Real, Test on Real (TRTR)** baseline: how well a model trained on real data does.
- **Train on Synthetic, Test on Real (TSTR)** utility: how well a model trained on synthetic data does.

The score for each model is $\text{TSTR} - \text{TRTR}$. **Closer to 0 is better.** Negative means synthetic training is worse; positive (rare) means it is better.

Two evaluations:

1. **Cross-validation** — 5 folds of the real training set. Each fold trains on real vs synthetic and tests on the held-out real fold.
2. **Holdout** — train on the *full* real and synthetic sets (no CV), test on `real_holdout_data`. This is the stricter check.

Categorical columns (including the label) must be encoded first. This notebook already ran the SynthEval preprocessor (ordinal encode + min-max scale), so `do_preprocess=False`. The label is passed separately and is **not** listed in `numerical_columns` / `categorical_columns`.

### Metric results

Headline numbers (average over the four models):

- **`mean_f1_difference`**: mean of $(\text{TSTR} - \text{TRTR})$ from CV. Near 0 is best.
- **`f1_difference_standard_error`**: how much that gap varies across models.
- **`mean_f1_difference_holdout`** / **`f1_difference_standard_error_holdout`**: same, but on the holdout set.

Per-model F1 (higher is better for each number; the *gap* is what matters):

- `*_real_train_f1` — TRTR on CV folds
- `*_synthetic_train_f1` — TSTR on CV folds
- `*_real_train_f1_holdout` / `*_synthetic_train_f1_holdout` — same on holdout



In [13]:
from midst_toolkit.common.enumerations import TaskType

from implementations.tabular_data.evaluation.preprocessing import remove_label_column_from_other_columns


TASK_TYPE = "multiclass"
LABEL_COLUMN = "trans_type"
# Explicitly removing the target/label column from other column names
filtered_numerical_columns, filtered_categorical_columns = remove_label_column_from_other_columns(
    LABEL_COLUMN,
    numerical_columns,
    categorical_columns,
)
assert TASK_TYPE != TaskType.REGRESSION, "F1 Score Difference is only supported for classification tasks"
log(INFO, "Running F1 Score Difference Evaluation")
metric = MeanF1ScoreDifference(
    categorical_columns=filtered_categorical_columns,
    numerical_columns=filtered_numerical_columns,
    label_column=LABEL_COLUMN,
    folds=5,
    f1_type="macro",
    # Already preprocessing above
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data, syntheval_real_data_holdout)
log_metrics("F1 Score Difference", results)

INFO :      Running F1 Score Difference Evaluation
INFO :      
F1 Score Difference
--------------------------------------------------------------------------------

INFO :      Metric: random_forest_real_train_f1	Score: 0.8078533176364469
INFO :      Metric: random_forest_synthetic_train_f1	Score: 0.6972910884541382
INFO :      Metric: adaboost_real_train_f1	Score: 0.6442279345908849
INFO :      Metric: adaboost_synthetic_train_f1	Score: 0.6540779214108208
INFO :      Metric: svm_real_train_f1	Score: 0.661935760004855
INFO :      Metric: svm_synthetic_train_f1	Score: 0.6613583436607138
INFO :      Metric: logreg_real_train_f1	Score: 0.5842187408567382
INFO :      Metric: logreg_synthetic_train_f1	Score: 0.5864124789709388
INFO :      Metric: mean_f1_difference	Score: -0.02477398014807833
INFO :      Metric: f1_difference_standard_error	Score: 0.0036081493194391498
INFO :      Metric: random_forest_real_train_f1_holdout	Score: 0.6627851549199865
INFO :      Metric: random_forest_synthe

## Utility: Regression Score Difference

The F1 metric above is for **classification**. This section is the same idea for a **numeric** target: can a regressor trained on synthetic rows predict a real value as well as one trained on real rows?

Here the label is `balance`. Four sklearn models are trained twice — `LinearRegression`, `MLPRegressor`, `XGBRegressor`, and `RandomForestRegressor` — once on **real** data and once on **synthetic** data. Both versions are then scored on the same **real holdout** set (`real_holdout_data`). A holdout set is **required**.

- **Train on Real, Test on Real (TRTR)** baseline: how well a model trained on real data predicts `balance` on holdout rows.
- **Train on Synthetic, Test on Real (TSTR)** utility: how well a model trained on synthetic data does on those same holdout rows.

If the synthetic table is a good substitute for the real table, TSTR should match TRTR.

### How training works

The models and their hyper-parameter grids come from `regression_config.yaml` (or the toolkit default JSON). For each model:

1. Split the training table (real or synthetic) into a fit set and a **validation** set.
2. If the config lists several hyper-parameter combinations, run a **grid search** on that validation split.
3. Pick the best parameters **separately for each metric** (best $R^2$ need not be the same as best MAE).
4. Retrain with those parameters and score on the holdout set.

Four holdout metrics are recorded:

- **$R^2$**: fraction of variance in `balance` explained by the model. $1$ is perfect; $0$ is no better than predicting the mean; **negative** means worse than the mean.
- **Explained variance**: similar to $R^2$; close to $1$ is better.
- **MAE** (mean absolute error): average absolute miss. **Smaller is better.**
- **MSE** (mean squared error): average squared miss (penalizes large errors more). **Smaller is better.**

### Preprocessing

This metric uses its **own** pipeline (`do_preprocess=True`), not the SynthEval preprocessor used above. Independently on real and synthetic (and the holdout, in the same way):

- numerical features → MinMax scaling
- categorical features → One-Hot encoding

The label is **not** transformed unless `preprocess_labels=True` (this notebook turns that on so `balance` is scaled). The label must be a **float** column and must **not** appear in `numerical_columns` / `categorical_columns`.

### Metric results

Every difference is $\text{TSTR} - \text{TRTR}$ (synthetic-trained score minus real-trained score). **Closer to 0 is better**, but the *sign* you want depends on the metric:

- **$R^2$ and explained variance**: a **non-negative** difference is good (synthetic training as good as, or better than, real). A large negative value means synthetic training is worse.
- **MAE and MSE**: a **non-positive** difference is good (synthetic training has equal or smaller error). A positive difference means synthetic training is worse.

Headline numbers (average over the four models):

- **`avg_r2_difference`** / **`avg_explained_variance_difference`**: mean TSTR−TRTR for those metrics. Near $0$ from above is best.
- **`avg_mean_absolute_error_difference`** / **`avg_mean_squared_error_difference`**: mean TSTR−TRTR for the error metrics. Near $0$ from below is best.
- **`avg_all_scores_difference`**: mean of *all* metric values across all models, then TSTR−TRTR. This mixes $R^2$-style scores with errors, so treat it as a rough summary, not the main number.

Per-model scores (`include_additional_metrics=True` in this notebook). Replace `{Model}` with `LinearRegression`, `MLPRegressor`, `XGBRegressor`, or `RandomForestRegressor`:

- `{Model}_{metric}_difference` — TSTR − TRTR for that model and metric
- `real_{Model}_{metric}` — TRTR (train real, test holdout)
- `synthetic_{Model}_{metric}` — TSTR (train synthetic, test holdout)

The same pattern exists for averages: `real_avg_r2`, `synthetic_avg_r2`, `real_avg_all_scores`, `synthetic_avg_all_scores`, and so on.

If `include_additional_metrics` were `False`, only the `*_difference` keys would be returned.


In [14]:
# Before running the next cell, make sure to add the previous label column back to
# the categorical columns. This is because we'll change the label column to a
# numerical column (for regression tasks)
if "trans_type" not in categorical_columns:
    categorical_columns.append("trans_type")
# Remove any repeated columns
categorical_columns = list(set(categorical_columns))
numerical_columns = list(set(numerical_columns))
print(categorical_columns)
print(numerical_columns)

['k_symbol', 'bank', 'operation', 'trans_type']
['account', 'amount', 'trans_date', 'balance']


In [15]:
from midst_toolkit.common.enumerations import TaskType

from implementations.tabular_data.evaluation.preprocessing import remove_label_column_from_other_columns


TASK_TYPE = "regression"
# Make sure that the label column is a numerical column
LABEL_COLUMN = "balance"
# Specify the regression models' parameters and structure in a config file.
# Make sure it's a PATH object.
REGRESSOIN_CONFIG_PATH = Path("implementations/tabular_data/evaluation/quality/regression_config.yaml")
# Explicitly removing the target/label column from other column names
filtered_numerical_columns, filtered_categorical_columns = remove_label_column_from_other_columns(
    LABEL_COLUMN,
    numerical_columns,
    categorical_columns,
)
log(INFO, "Running Regression Score Difference Evaluation")
metric = MeanRegressionDifference(
    categorical_columns=filtered_categorical_columns,
    numerical_columns=filtered_numerical_columns,
    label_column=LABEL_COLUMN,
    preprocess_labels=True,
    include_additional_metrics=True,
    # Regression has it's own preprocessing pipeline
    do_preprocess=True,
    regressors_config=REGRESSOIN_CONFIG_PATH,
    measure_metrics_in_original_label_space=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data, syntheval_real_data_holdout)
log_metrics("Regression Score Difference", results)

INFO :      Running Regression Score Difference Evaluation
INFO :      Default preprocessing will be performed during computation.
100%|██████████| 1/1 [00:00<00:00, 45.37it/s]
INFO :      Getting Regressor: LinearRegression
INFO :      Getting Regressor: LinearRegression
INFO :      Getting Regressor: LinearRegression
INFO :      Getting Regressor: LinearRegression
100%|██████████| 12/12 [00:13<00:00,  1.10s/it]
INFO :      Getting Regressor: MLPRegressor
INFO :      Getting Regressor: MLPRegressor
INFO :      Getting Regressor: MLPRegressor
INFO :      Getting Regressor: MLPRegressor
  0%|          | 0/36 [00:00<?, ?it/s]INFO :      Getting Regressor: XGBRegressor
INFO :      Getting Regressor: XGBRegressor
  6%|▌         | 2/36 [00:00<00:02, 14.85it/s]INFO :      Getting Regressor: XGBRegressor
INFO :      Getting Regressor: XGBRegressor
 11%|█         | 4/36 [00:00<00:02, 15.86it/s]INFO :      Getting Regressor: XGBRegressor
INFO :      Getting Regressor: XGBRegressor
 17%|█▋      

In [16]:
# Added — Inspect Regression Evaluation Results

import pprint

pprint.pprint(results)

{'LinearRegression_explained_variance_difference': -0.0036866497970460266,
 'LinearRegression_mean_absolute_error_difference': 0.0005302151633684582,
 'LinearRegression_mean_squared_error_difference': 4.013784921645322e-05,
 'LinearRegression_r2_difference': -0.0037094925357296082,
 'MLPRegressor_explained_variance_difference': -0.010163235433276574,
 'MLPRegressor_mean_absolute_error_difference': -0.0006512844468883339,
 'MLPRegressor_mean_squared_error_difference': 0.00023479572139579324,
 'MLPRegressor_r2_difference': -0.00854903064166801,
 'RandomForestRegressor_explained_variance_difference': 0.0020765038962219373,
 'RandomForestRegressor_mean_absolute_error_difference': -0.00026300132565793,
 'RandomForestRegressor_mean_squared_error_difference': -2.1901690352926252e-05,
 'RandomForestRegressor_r2_difference': 0.0003933996435159237,
 'XGBRegressor_explained_variance_difference': -0.009765386581420898,
 'XGBRegressor_mean_absolute_error_difference': 0.0006425939500331879,
 'XGBReg

In [17]:
import pandas as pd

metrics = [
    "avg_r2_difference",
    "avg_mean_absolute_error_difference",
    "avg_mean_squared_error_difference",
    "avg_explained_variance_difference",
]

summary = pd.DataFrame({
    "Metric": metrics,
    "Difference": [results[m] for m in metrics]
})

display(summary)

,Metric,Difference
0,avg_r2_difference,-0.005415
1,avg_mean_absolute_error_difference,0.000065
2,avg_mean_squared_error_difference,0.000090
3,avg_explained_variance_difference,-0.005385


In [18]:
# Rare Category Preservation — k_symbol

column = "k_symbol"
rare_threshold = 0.05

real_freq = real_train_data[column].value_counts(
    normalize=True, dropna=False
)

synthetic_freq = synthetic_data[column].value_counts(
    normalize=True, dropna=False
)

comparison = pd.DataFrame({
    "Real Frequency": real_freq,
    "Synthetic Frequency": synthetic_freq
}).fillna(0)

comparison["Absolute Difference"] = (
    comparison["Synthetic Frequency"] -
    comparison["Real Frequency"]
).abs()

rare_categories = comparison[
    comparison["Real Frequency"] < rare_threshold
].sort_values("Real Frequency")

display(rare_categories)

,Real Frequency,Synthetic Frequency,Absolute Difference
k_symbol,,,
4,0.001187,0.000562,0.000625
8,0.012250,0.011687,0.000563
3,0.019062,0.003375,0.015688
2,0.027625,0.029937,0.002312


In [19]:
# High Transaction Amount Preservation

thresholds = {
    "P95": real_train_data["amount"].quantile(0.95),
    "P99": real_train_data["amount"].quantile(0.99)
}

rows = []

for name, threshold in thresholds.items():
    real_rate = (real_train_data["amount"] > threshold).mean()
    synthetic_rate = (synthetic_data["amount"] > threshold).mean()
    holdout_rate = (real_holdout_data["amount"] > threshold).mean()

    rows.append({
        "Threshold": name,
        "Amount Cutoff": threshold,
        "Real Train Rate": real_rate,
        "Real Holdout Rate": holdout_rate,
        "Synthetic Rate": synthetic_rate,
        "Tail Mass Ratio": (
            synthetic_rate / real_rate
            if real_rate > 0 else float("nan")
        )
    })

tail_comparison = pd.DataFrame(rows)
display(tail_comparison)

,Threshold,Amount Cutoff,Real Train Rate,Real Holdout Rate,Synthetic Rate,Tail Mass Ratio
0,P95,25762.95,0.05,0.05050,0.04975,0.995
1,P99,45661.25,0.01,0.01225,0.01050,1.050


In [ ]:
# Rare Category × High Transaction Amount

rare_values = rare_categories.index.tolist()
p95_cutoff = real_train_data["amount"].quantile(0.95)

rows = []

for category in rare_values:
    real = real_train_data[real_train_data["k_symbol"] == category]
    synth = synthetic_data[synthetic_data["k_symbol"] == category]

    real_high = (real["amount"] > p95_cutoff).sum()
    synth_high = (synth["amount"] > p95_cutoff).sum()

    rows.append({
        "Category": category,
        "Real Count": len(real),
        "Synthetic Count": len(synth),
        "Real High Count": real_high,
        "Synthetic High Count": synth_high,
        "Real High Rate": real_high / len(real) if len(real) else float("nan"),
        "Synthetic High Rate": synth_high / len(synth) if len(synth) else float("nan")
    })

rare_high_comparison = pd.DataFrame(rows)
display(rare_high_comparison)

,Category,Real Count,Synthetic Count,Real High Count,Synthetic High Count,Real High Rate,Synthetic High Rate
0,4,19,9,0,0,0.0,0.0
1,8,196,187,0,0,0.0,0.0
2,3,305,54,0,0,0.0,0.0
3,2,442,479,0,0,0.0,0.0


In [21]:
# Category-Specific Tail Preservation

rows = []

for category in rare_values:
    real = real_train_data.loc[
        real_train_data["k_symbol"] == category, "amount"
    ].dropna()

    synth = synthetic_data.loc[
        synthetic_data["k_symbol"] == category, "amount"
    ].dropna()

    for q in [0.90, 0.95]:
        cutoff = real.quantile(q)

        real_rate = (real > cutoff).mean()
        synth_rate = (synth > cutoff).mean()

        rows.append({
            "Category": category,
            "Quantile": q,
            "Real Count": len(real),
            "Synthetic Count": len(synth),
            "Real Cutoff": cutoff,
            "Real Tail Rate": real_rate,
            "Synthetic Tail Rate": synth_rate,
            "Tail Mass Ratio": (
                synth_rate / real_rate
                if real_rate > 0 else float("nan")
            )
        })

category_tail = pd.DataFrame(rows)
display(category_tail)

,Category,Quantile,Real Count,Synthetic Count,Real Cutoff,Real Tail Rate,Synthetic Tail Rate,Tail Mass Ratio
0,4,0.90,19,9,85.500,0.105263,0.666667,6.333333
1,4,0.95,19,9,103.590,0.052632,0.666667,12.666667
2,8,0.90,196,187,7360.900,0.102041,0.085561,0.838503
3,8,0.95,196,187,8127.975,0.051020,0.042781,0.838503
4,3,0.90,305,54,4315.400,0.101639,0.055556,0.546595
5,3,0.95,305,54,6079.400,0.052459,0.018519,0.353009
6,2,0.90,442,479,6725.300,0.101810,0.073069,0.717699
7,2,0.95,442,479,6860.000,0.049774,0.039666,0.796925


In [22]:
# Holdout Validation for Category-Specific Tails

rows = []

for category in rare_values:
    real = real_train_data.loc[
        real_train_data["k_symbol"] == category, "amount"
    ].dropna()

    holdout = real_holdout_data.loc[
        real_holdout_data["k_symbol"] == category, "amount"
    ].dropna()

    synth = synthetic_data.loc[
        synthetic_data["k_symbol"] == category, "amount"
    ].dropna()

    for q in [0.90, 0.95]:
        cutoff = real.quantile(q)

        rows.append({
            "Category": category,
            "Quantile": q,
            "Train Count": len(real),
            "Holdout Count": len(holdout),
            "Synthetic Count": len(synth),
            "Train Tail Rate": (real > cutoff).mean(),
            "Holdout Tail Rate": (holdout > cutoff).mean(),
            "Synthetic Tail Rate": (synth > cutoff).mean()
        })

holdout_tail = pd.DataFrame(rows)
display(holdout_tail)

,Category,Quantile,Train Count,Holdout Count,Synthetic Count,Train Tail Rate,Holdout Tail Rate,Synthetic Tail Rate
0,4,0.90,19,10,9,0.105263,0.100000,0.666667
1,4,0.95,19,10,9,0.052632,0.000000,0.666667
2,8,0.90,196,46,187,0.102041,0.130435,0.085561
3,8,0.95,196,46,187,0.051020,0.108696,0.042781
4,3,0.90,305,61,54,0.101639,0.081967,0.055556
5,3,0.95,305,61,54,0.052459,0.049180,0.018519
6,2,0.90,442,118,479,0.101810,0.118644,0.073069
7,2,0.95,442,118,479,0.049774,0.050847,0.039666


In [23]:
column = "k_symbol"

freq_comparison = pd.DataFrame({
    "Real Train": real_train_data[column].value_counts(normalize=True),
    "Real Holdout": real_holdout_data[column].value_counts(normalize=True),
    "Synthetic": synthetic_data[column].value_counts(normalize=True)
}).fillna(0)

display(freq_comparison.loc[rare_values])

,Real Train,Real Holdout,Synthetic
k_symbol,,,
4,0.001187,0.00250,0.000562
8,0.012250,0.01150,0.011687
3,0.019062,0.01525,0.003375
2,0.027625,0.02950,0.029937
